# NutriPredict Congo (Vérification des données DHS)
**Auteur :** MPOY Schekina Lutte-de-vie  
**Date :** 22 juin 2026  
**Notebook :** Vérification initiale des fichiers DHS (01 / 05 )

---

## Objectif de ce notebook

Avant tout nettoyage ou modélisation, ce notebook vérifie que :
1. Les fichiers DHS téléchargés sont lisibles en Python
2. Les variables clés du projet sont bien présentes (hw70, hw71, hw72, hv025, hv270)
3. Le volume réel de données disponibles est suffisant pour la suite

Ce notebook ne modifie aucune donnée il observe uniquement.

---

## Fichiers utilisés

| Fichier | Contenu | Pays |
|---|---|---|
| CGKR61FL.DTA | Children's Recode — données enfants | Congo 2011-12 |
| CGHR61FL.DTA | Household Recode — données ménages | Congo 2011-12 |

*Les fichiers RDC et Cameroun seront vérifiés dans la section 3 de ce notebook.*

In [1]:

import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Vérification des versions
print(f"pandas : {pd.__version__}")
print(f"Python : {os.sys.version}")

pandas : 3.0.3
Python : 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]


## 1. Chargement des fichiers Congo 2011-2012

On charge les deux fichiers DHS du Congo séparément avant toute fusion.  
`convert_categoricals=False` est obligatoire : sans ça, pandas convertit 
les codes numériques en texte catégoriel, ce qui complique le nettoyage 
ultérieur. On garde les valeurs brutes et on décode manuellement ce dont 
on a besoin.

In [8]:
# Recupération du chemins des données
BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR   = os.path.join(BASE_DIR, 'data', 'raw', 'congo2011_12')

CHEMIN_KR = os.path.join(RAW_DIR, 'CGKR61DT', 'CGKR61FL.DTA')
CHEMIN_HR = os.path.join(RAW_DIR, 'CGHR61DT', 'CGHR61FL.DTA')

# Vérification
print(f"Racine projet : {BASE_DIR}")
print(f"Dossier raw   : {RAW_DIR}")
print()
for nom, chemin in [("KR", CHEMIN_KR), ("HR", CHEMIN_HR)]:
    if os.path.exists(chemin):
        taille = os.path.getsize(chemin) / (1024*1024)
        print(f" {nom} trouvé : {taille:.1f} Mo")
    else:
        print(f" {nom} INTROUVABLE")
        print(f"  Chemin testé : {chemin}")

Racine projet : c:\Users\HP\Downloads\NutriPredict-Congo
Dossier raw   : c:\Users\HP\Downloads\NutriPredict-Congo\data\raw\congo2011_12

 KR trouvé : 11.3 Mo
 HR trouvé : 40.7 Mo


## 2. Chargement et inspection du Children's Recode (KR)

Le fichier KR est le fichier central du projet.  
Chaque ligne correspond à un enfant de moins de 5 ans enquêté.  
C'est ici que se trouvent les variables cibles hw70, hw71, hw72.

In [3]:
# Chargement du fichier Children's Recode
df_kr = pd.read_stata(CHEMIN_KR, convert_categoricals=False)

print(f"Nombre de lignes  : {len(df_kr)}")
print(f"Nombre de colonnes : {len(df_kr.columns)}")

Nombre de lignes  : 9329
Nombre de colonnes : 1110


## 3. Vérification des variables clés
On vérifie que les variables indispensables au projet sont bien 
présentes dans le fichier KR, avant d'aller plus loin.

Variables cibles (Z-scores malnutrition) : hw70, hw71, hw72  
Variables démographiques enfant : b4, b8, bord  
Clés de jointure avec le fichier ménage : v001, v002

In [4]:
# Vérification des variables clés dans le fichier KR
variables_cles_kr = ['v001', 'v002', 'hw70', 'hw71', 'hw72', 'b4', 'b8', 'bord']

print("=== Présence des variables clés ===")
for var in variables_cles_kr:
    if var in df_kr.columns:
        manquants = df_kr[var].isna().sum()
        pct = manquants / len(df_kr) * 100
        print(f" {var:<8} | valeurs manquantes : {manquants:>4} ({pct:.1f}%)")
    else:
        print(f" {var} — ABSENTE du fichier")

print(f"\n=== Aperçu des 5 premières lignes ===")
print(df_kr[variables_cles_kr].head())

=== Présence des variables clés ===
 v001     | valeurs manquantes :    0 (0.0%)
 v002     | valeurs manquantes :    0 (0.0%)
 hw70     | valeurs manquantes : 4708 (50.5%)
 hw71     | valeurs manquantes : 4708 (50.5%)
 hw72     | valeurs manquantes : 4708 (50.5%)
 b4       | valeurs manquantes :    0 (0.0%)
 b8       | valeurs manquantes :  472 (5.1%)
 bord     | valeurs manquantes :    0 (0.0%)

=== Aperçu des 5 premières lignes ===
   v001  v002  hw70   hw71   hw72  b4   b8  bord
0     1     2 -67.0   68.0  167.0   2  4.0     4
1     1     4  36.0 -108.0 -200.0   1  0.0     1
2     1     5   NaN    NaN    NaN   1  1.0     4
3     1     5   NaN    NaN    NaN   2  3.0     3
4     1     6  35.0  -44.0  -78.0   2  0.0     1


## 4. Analyse des valeurs manquantes sur les Z-scores

50.5% de valeurs manquantes sur hw70/hw71/hw72 est attendu :  
les mesures anthropométriques ne sont prises que sur les enfants  
de moins de 5 ans présents au moment de l'enquête.  

Le dataset effectif pour la modélisation sera d'environ 4 621 lignes  
— ce qui reste suffisant pour un modèle de classification robuste.  

Les lignes sans mesures anthropométriques seront exclues lors du  
nettoyage (Phase 2), pas ici. Ce notebook observe uniquement.

In [5]:
# Distribution des valeurs brutes de hw70
# (sur les lignes où hw70 est renseigné uniquement)

hw70_valide = df_kr["hw70"].dropna()

# Statistique récapitulatif
print(f"median   {hw70_valide.median():.1f} ")
print(hw70_valide.describe().round(1))


median   -109.0 
count    4621.0
mean      206.1
std      1775.1
min      -585.0
25%      -204.0
50%      -109.0
75%       -13.0
max      9999.0
Name: hw70, dtype: float64


In [ ]:
# Détection des codes Aberants(Valeurs aberants): 

code_abberants = hw70_valide[hw70_valide>=9000]
print(f"Nombres : {len(code_abberants)}")
if len(code_abberants)>0:
    print(f"{code_abberants.unique()}")
else: 
    print(f"Acune valeurs abérantes détecté")

Nombres : 146
[9999. 9998. 9996.]


## 5. Chargement et inspection du Household Recode (HR)

Le fichier HR contient les caractéristiques socio-économiques  
de chaque ménage enquêté. Il sera fusionné avec le fichier KR  
via les clés v001/v002 (KR) et hv001/hv002 (HR).

Variables clés attendues :
- hv025 : milieu de résidence (1=urbain, 2=rural)
- hv270 : indice de richesse (quintiles 1 à 5)
- hv201 : source d'eau potable
- hv205 : type d'assainissement
- hv206 : accès à l'électricité
- hv219 : sexe du chef de ménage

In [7]:
#Chargement du fichier Household Recode
df_hr = pd.read_stata(CHEMIN_HR, convert_categoricals=False)

print(f"Nombre de lignes: {df_hr.shape[0]} \nNombre de colonnes : {df_hr.shape[1]}")

Nombre de lignes: 11632 
Nombre de colonnes : 3194


## 6. Vérification des variables clés du fichier HR

On vérifie la présence et le taux de valeurs manquantes  
sur les variables socio-économiques qui serviront de features  
dans le modèle, ainsi que sur les clés de jointure hv001/hv002.

In [34]:
# Vérification des variables clés de HR
variables_cles_hr =  ['hv001', 'hv002', 'hv025', 'hv270', 
                     'hv201', 'hv205', 'hv206', 'hv219']

for var in variables_cles_hr:
    if var in df_hr.columns:
        manquantes = df_hr[var].isna().sum()
        pct = manquants/len(df_hr)*100
    print(f"{var:<8}|Valeurs manquantes:{manquantes:<4} | Pourcentage : ({pct:.1f}%)")

hv001   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv002   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv025   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv270   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv201   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv205   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv206   |Valeurs manquantes:0    | Pourcentage : (0.0%)
hv219   |Valeurs manquantes:0    | Pourcentage : (0.0%)


In [14]:
# Aperçu des 5 premieres lignes 
print(df_hr[variables_cles_hr].head())

   hv001  hv002  hv025  hv270  hv201  hv205  hv206  hv219
0      1      1      2      1     21     23      0      2
1      1      2      2      4     21     22      1      1
2      1      3      2      1     42     23      0      1
3      1      4      2      2     21     23      0      1
4      1      5      2      1     21     23      0      1


## 7. Vérification de la compatibilité des clés de jointure

Avant toute fusion, on vérifie que les identifiants cluster/ménage  
(v001/v002 dans KR et hv001/hv002 dans HR) sont cohérents entre  
les deux fichiers, même plage de valeurs.

In [ ]:
# Vérification des clés de jointure 

# Clusters unique dans chaque fichiers
clusters_kr = set(df_kr["v001"].unique())
clusters_hr = set(df_hr["hv001"].unique())


# nombre de cluster présent :
print(f"Nombre de cluster dans KR : {len(clusters_kr)}")
print(f"Nombre de clusters dans hr : {len(clusters_hr)}")
print(f"Nombres de cluster Kr et non dans HR : { len(clusters_kr - clusters_hr)}")
print(f"Nombres de cluster HR et non dans KR : { len(clusters_hr - clusters_kr)}")

Nombre de cluster dans KR : 384
Nombre de clusters dans hr : 384
Nombres de cluster Kr et non dans HR : 0
Nombres de cluster HR et non dans KR : 0


In [36]:
# Vérification des ménages unique
df_kr["cles_menages"] = df_kr["v001"].astype("str")+"_"+df_kr["v002"].astype("str")
df_hr["cles_menages"] = df_hr["hv001"].astype("str")+"_"+df_hr["hv002"].astype("str")

# nombres de ménages présent :
print(f"Nombre de ménages dans KR : {len(df_kr["cles_menages"])}")
print(f"Nombre de ménages dans HR : {len(df_hr["cles_menages"])}")
print(f"Nombres de ménages présent dans KR et non dans HR : {len(set(df_kr["cles_menages"]) - set(df_hr["cles_menages"]))}")
print(f"Nombres de ménages présent dans HR et non dans KR : {len(set(df_hr["cles_menages"]) - set(df_kr["cles_menages"]))}")

Nombre de ménages dans KR : 9329
Nombre de ménages dans HR : 11632
Nombres de ménages présent dans KR et non dans HR : 0
Nombres de ménages présent dans HR et non dans KR : 5800


## 8. Synthèse de la vérification

### Ce qu'on a confirmé

| Fichier | Lignes | Colonnes | Statut |
|---|---|---|---|
| CGKR61FL.DTA (Children's Recode) | 9 329 | 1 110 | Lisible |
| CGHR61FL.DTA (Household Recode) | 11 632 | 3 194 | Lisible |

### Points clés à retenir pour la Phase 2

- **Dataset effectif pour la modélisation : ~4 621 enfants**
  (9 329 lignes totales dont 50.5% sans mesures anthropométriques)
- **146 codes aberrants DHS** sur hw70 (valeurs 9996, 9998, 9999)
   à remplacer par NaN en Phase 2 avant tout calcul
- **Clés de jointure complètes** : 0 cluster absent, 0 ménage KR
  sans correspondance dans HR, la fusion sera propre
- **5 800 ménages HR sans enfant de moins de 5 ans** :
ces ménages disparaîtront naturellement lors du merge